# Operational tables and figures for the IEEE article

This notebook reads the raw artifacts from the Original trace campaign and the nine fixed-FPS configurations. It generates the article tables, vector figures, captions, and audit files under `article_artifacts/`.

The experimental replicate is the complete run (`n=5` per configuration). Passage-level observations are aggregated within each run before confidence intervals are calculated. The Original trace is kept as a separate, irregular-timestamp input and is never treated as a fixed FPS level or included in fixed-FPS comparisons.

All numerical outputs are generated from `metrics.json` and the raw CPU, RAM, temperature, and power CSV files. No value is copied from Markdown reports.

In [ ]:
from __future__ import annotations

import json
import math
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from typing import Any

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "power_runs" / "battery_mas-single_20260708_104924").exists():
            return candidate
    raise FileNotFoundError("Repository root containing the requested campaigns was not found.")


REPO_ROOT = find_repo_root(Path.cwd())
FIXED_ROOT = REPO_ROOT / "power_runs" / "battery_mas-single_20260708_104924"
ORIGINAL_ROOT = REPO_ROOT / "power_runs" / "battery_mas-single_native_20260713_185901"
REFERENCE_NOTEBOOKS = [
    REPO_ROOT / "data-analysis" / "analise_completa_5runs.ipynb",
    REPO_ROOT / "data-analysis" / "baseline_5runs_analysis_corrected.ipynb",
]
RESIDUAL_MODULE_PATH = FIXED_ROOT / "residual_inference_workload.py"

OUTPUT_ROOT = REPO_ROOT / "article_artifacts"
NOTEBOOK_DIR = OUTPUT_ROOT / "notebook"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
for directory in [NOTEBOOK_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(FIXED_ROOT))
from residual_inference_workload import (  # audited percentile and tolerance
    LATENCY_IDENTITY_ATOL_S,
    _p95 as audited_p95,
)

FIXED_FPS = [1, 2, 3, 4, 5, 10, 15, 20, 30]
RUN_NUMBERS = [1, 2, 3, 4, 5]
CONFIG_ORDER = ["Original", *map(str, FIXED_FPS)]
EXPECTED_PASSAGES = 184
T_CRIT_95 = float(stats.t.ppf(0.975, 4))
SYMLOG_LINTHRESH = 0.1

COLORS = {
    "mean": "#0072B2",
    "p95": "#E69F00",
    "maximum": "#D55E00",
    "original": "#666666",
    "original_fill": "#E5E5E5",
    "grid": "#D9D9D9",
    "box_fill": "#DCEAF4",
    "box_edge": "#0072B2",
    "median": "#222222",
    "outliers": "#777777",
}
STYLES = {
    "mean": dict(color=COLORS["mean"], linestyle="-", marker="o"),
    "p95": dict(color=COLORS["p95"], linestyle="--", marker="s"),
    "maximum": dict(color=COLORS["maximum"], linestyle=":", marker="^"),
}


def available_font() -> str:
    for family in ["Times New Roman", "STIXGeneral", "DejaVu Serif"]:
        path = font_manager.findfont(family, fallback_to_default=False)
        if Path(path).exists():
            return family
    return "DejaVu Serif"


SELECTED_FONT = available_font()
mpl.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": [SELECTED_FONT, "STIXGeneral", "DejaVu Serif"],
        "font.size": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "axes.labelsize": 8.5,
        "legend.fontsize": 8,
        "axes.linewidth": 0.7,
        "lines.linewidth": 1.4,
        "lines.markersize": 4.5,
        "grid.color": COLORS["grid"],
        "grid.linewidth": 0.6,
        "grid.alpha": 1.0,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.dpi": 300,
    }
)


def ci95(values: pd.Series | np.ndarray) -> dict[str, float]:
    x = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(float)
    n = len(x)
    mean = float(x.mean()) if n else math.nan
    std = float(x.std(ddof=1)) if n > 1 else math.nan
    half = T_CRIT_95 * std / math.sqrt(n) if n > 1 else math.nan
    return {
        "n": n,
        "mean": mean,
        "std": std,
        "ci95_low": mean - half if n > 1 else math.nan,
        "ci95_high": mean + half if n > 1 else math.nan,
        "ci95_half": half,
    }


def p95(values: pd.Series | np.ndarray) -> float:
    return audited_p95(pd.Series(values))


def save_figure(fig: mpl.figure.Figure, stem: str) -> None:
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", facecolor="white")
    fig.savefig(FIGURE_DIR / f"{stem}.png", dpi=300, facecolor="white")
    plt.close(fig)


def panel_label(ax: mpl.axes.Axes, label: str) -> None:
    ax.text(0.015, 0.97, label, transform=ax.transAxes, ha="left", va="top", fontsize=9, fontweight="bold")


def style_axis(ax: mpl.axes.Axes) -> None:
    ax.grid(axis="y")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def markdown_table(frame: pd.DataFrame) -> str:
    headers = [str(c) for c in frame.columns]
    rows = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for values in frame.astype(str).itertuples(index=False, name=None):
        rows.append("| " + " | ".join(values) + " |")
    return "\n".join(rows)


print("Repository:", REPO_ROOT)
print("Output:", OUTPUT_ROOT)
print("Selected font:", SELECTED_FONT)
print(f"t critical (95%, df=4): {T_CRIT_95:.6f}")

In [ ]:
FIXED_DIR_RE = re.compile(r"mas-single_(?P<fps>\d+)fps_r(?P<run>\d+)$")
ORIGINAL_DIR_RE = re.compile(r"mas-single_native_r(?P<run>\d+)$")


def unique_child_file(run_dir: Path, filename: str) -> Path:
    matches = sorted(run_dir.glob(f"*/{filename}"))
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one {filename} below {run_dir}; found {len(matches)}")
    return matches[0]


run_specs: list[dict[str, Any]] = []
for run_dir in sorted(FIXED_ROOT.glob("mas-single_*fps_r*")):
    match = FIXED_DIR_RE.fullmatch(run_dir.name)
    if not match:
        continue
    fps, run = int(match.group("fps")), int(match.group("run"))
    if fps not in FIXED_FPS or run not in RUN_NUMBERS:
        continue
    metrics_path = unique_child_file(run_dir, "metrics.json")
    metrics_dir = metrics_path.parent
    run_specs.append(
        {
            "configuration": str(fps),
            "fps": float(fps),
            "run_number": run,
            "run": f"r{run}",
            "campaign": "Fixed FPS",
            "run_dir": run_dir,
            "metrics_path": metrics_path,
            "cpu_path": metrics_dir / "cpu.csv",
            "mem_path": metrics_dir / "mem.csv",
            "temp_path": metrics_dir / "temp.csv",
            "power_path": run_dir / "power.csv",
        }
    )

for run_dir in sorted(ORIGINAL_ROOT.glob("mas-single_native_r*")):
    match = ORIGINAL_DIR_RE.fullmatch(run_dir.name)
    if not match:
        continue
    run = int(match.group("run"))
    if run not in RUN_NUMBERS:
        continue
    metrics_path = unique_child_file(run_dir, "metrics.json")
    metrics_dir = metrics_path.parent
    run_specs.append(
        {
            "configuration": "Original",
            "fps": math.nan,
            "run_number": run,
            "run": f"r{run}",
            "campaign": "Original trace",
            "run_dir": run_dir,
            "metrics_path": metrics_path,
            "cpu_path": metrics_dir / "cpu.csv",
            "mem_path": metrics_dir / "mem.csv",
            "temp_path": metrics_dir / "temp.csv",
            "power_path": run_dir / "power.csv",
        }
    )

run_specs_df = pd.DataFrame(run_specs)
run_specs_df["configuration"] = pd.Categorical(run_specs_df["configuration"], CONFIG_ORDER, ordered=True)
run_specs_df = run_specs_df.sort_values(["configuration", "run_number"]).reset_index(drop=True)

required_paths = ["metrics_path", "cpu_path", "mem_path", "temp_path", "power_path"]
missing_files = [str(row[column]) for _, row in run_specs_df.iterrows() for column in required_paths if not Path(row[column]).is_file()]
if missing_files:
    raise FileNotFoundError("Missing raw input files:\n" + "\n".join(missing_files))

print("Runs discovered:", len(run_specs_df))
display(run_specs_df[["configuration", "run", "campaign", "metrics_path"]])

In [ ]:
def parse_timestamp(value: Any, field: str, context: dict[str, Any], issues: list[dict[str, Any]]) -> pd.Timestamp:
    try:
        parsed = pd.Timestamp(value)
    except Exception:
        parsed = pd.NaT
    if pd.isna(parsed):
        issues.append({**context, "issue": "missing_or_invalid_timestamp", "field": field, "value": value})
        return pd.NaT
    if getattr(parsed, "tzinfo", None) is not None:
        parsed = parsed.tz_localize(None)
    return parsed


passage_rows: list[dict[str, Any]] = []
metric_issues: list[dict[str, Any]] = []
run_windows: list[dict[str, Any]] = []

for spec in run_specs_df.itertuples(index=False):
    metrics_path = Path(spec.metrics_path)
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    animals = payload.get("animals")
    if not isinstance(animals, dict):
        raise ValueError(f"animals is not an object in {metrics_path}")
    base_run = {"configuration": str(spec.configuration), "run": spec.run}
    load_start = parse_timestamp(payload.get("load_model_start"), "load_model_start", base_run, metric_issues)
    run_final_times: list[pd.Timestamp] = []

    for animal_id, passage in animals.items():
        context = {**base_run, "animal_id": str(animal_id)}
        first_capture = parse_timestamp(passage.get("first_image_capture_time"), "first_image_capture_time", context, metric_issues)
        last_capture = parse_timestamp(passage.get("last_image_capture_time"), "last_image_capture_time", context, metric_issues)
        passage_final = parse_timestamp(passage.get("weight_prediction_final"), "passage.weight_prediction_final", context, metric_issues)
        if pd.notna(passage_final):
            run_final_times.append(passage_final)

        total_images = pd.to_numeric(passage.get("total_of_images"), errors="coerce")
        suitable_images = pd.to_numeric(passage.get("suitable_images"), errors="coerce")
        imgs = passage.get("imgs")
        if not isinstance(imgs, dict):
            metric_issues.append({**context, "issue": "invalid_imgs", "field": "imgs", "value": type(imgs).__name__})
            imgs = {}

        suitable_match = pd.notna(suitable_images) and int(suitable_images) == len(imgs)
        if not suitable_match:
            metric_issues.append(
                {**context, "issue": "suitable_images_imgs_mismatch", "field": "suitable_images/imgs", "value": f"{suitable_images}/{len(imgs)}"}
            )

        starts: list[pd.Timestamp] = []
        finals: list[pd.Timestamp] = []
        timestamp_valid = pd.notna(last_capture) and pd.notna(passage_final)
        prediction_order_ok = True
        for image_id, image in imgs.items():
            image_context = {**context, "image_id": str(image_id)}
            if not isinstance(image, dict):
                metric_issues.append({**image_context, "issue": "invalid_image_record", "field": "imgs", "value": type(image).__name__})
                timestamp_valid = False
                continue
            start = parse_timestamp(image.get("weight_prediction_start"), "imgs.weight_prediction_start", image_context, metric_issues)
            final = parse_timestamp(image.get("weight_prediction_final"), "imgs.weight_prediction_final", image_context, metric_issues)
            if pd.isna(start) or pd.isna(final):
                timestamp_valid = False
                continue
            starts.append(start)
            finals.append(final)
            if start > final:
                prediction_order_ok = False
                metric_issues.append({**image_context, "issue": "prediction_start_after_final", "field": "imgs", "value": f"{start.isoformat()} > {final.isoformat()}"})

        if timestamp_valid:
            active = sum(start <= last_capture < final for start, final in zip(starts, finals))
            not_started = sum(start > last_capture for start in starts)
            residual = active + not_started
            residual_by_final = sum(final > last_capture for final in finals)
            max_image_final = max(finals) if finals else last_capture
            prediction_drain_s = max(0.0, (max_image_final - last_capture).total_seconds())
            post_capture_latency_s = (passage_final - last_capture).total_seconds()
            finalization_overhead_s = (passage_final - max(last_capture, max_image_final)).total_seconds()
        else:
            active = not_started = residual = residual_by_final = pd.NA
            max_image_final = pd.NaT
            prediction_drain_s = post_capture_latency_s = finalization_overhead_s = math.nan

        passage_rows.append(
            {
                "configuration": str(spec.configuration),
                "fps": spec.fps,
                "run": spec.run,
                "run_number": spec.run_number,
                "animal_id": str(animal_id),
                "total_images": total_images,
                "suitable_images": suitable_images,
                "imgs_count": len(imgs),
                "suitable_images_matches_imgs": suitable_match,
                "first_capture": first_capture,
                "last_capture": last_capture,
                "passage_final": passage_final,
                "prediction_order_ok": prediction_order_ok,
                "active_inferences": active,
                "not_started_inferences": not_started,
                "residual_inferences": residual,
                "residual_by_prediction_final": residual_by_final,
                "has_residual": bool(residual > 0) if residual is not pd.NA else pd.NA,
                "prediction_drain_s": prediction_drain_s,
                "post_capture_latency_s": post_capture_latency_s,
                "finalization_overhead_s": finalization_overhead_s,
            }
        )

    run_end = max(run_final_times) if run_final_times else pd.NaT
    run_windows.append({"configuration": str(spec.configuration), "run": spec.run, "analysis_start": load_start, "analysis_end": run_end})

passage_df = pd.DataFrame(passage_rows)
for column in ["total_images", "suitable_images", "active_inferences", "not_started_inferences", "residual_inferences", "residual_by_prediction_final"]:
    passage_df[column] = passage_df[column].astype("Int64")
passage_df["has_residual"] = passage_df["has_residual"].astype("boolean")
metric_issues_df = pd.DataFrame(metric_issues)
run_windows_df = pd.DataFrame(run_windows)


def integrity_row(check: str, mask_or_value: Any, details: str, total: int | None = None) -> dict[str, Any]:
    if isinstance(mask_or_value, (pd.Series, np.ndarray, list)):
        violations = int(np.asarray(mask_or_value).sum())
        passed = violations == 0
    else:
        passed = bool(mask_or_value)
        violations = 0 if passed else (1 if total is None else total)
    return {"check": check, "passed": passed, "violations": violations, "details": details}


run_counts = passage_df.groupby(["configuration", "run"], observed=True).size()
runs_per_configuration = run_specs_df.groupby("configuration", observed=True)["run"].nunique().reindex(CONFIG_ORDER, fill_value=0)
duplicate_configuration_runs = run_specs_df.duplicated(["configuration", "run"], keep=False)
duplicate_metrics_paths = run_specs_df["metrics_path"].astype(str).duplicated(keep=False)
missing_timestamp_count = int(metric_issues_df.get("issue", pd.Series(dtype=str)).eq("missing_or_invalid_timestamp").sum())
latency_error = (
    passage_df["post_capture_latency_s"]
    - passage_df["prediction_drain_s"]
    - passage_df["finalization_overhead_s"]
).abs()

integrity_rows = [
    integrity_row("exactly five runs per configuration", runs_per_configuration.eq(5).all(), runs_per_configuration.to_dict()),
    integrity_row("exactly 184 passages per run", run_counts.eq(EXPECTED_PASSAGES).all(), f"min={run_counts.min()}, max={run_counts.max()}"),
    integrity_row("no duplicated metrics.json", ~(duplicate_configuration_runs | duplicate_metrics_paths).any(), f"paths={run_specs_df['metrics_path'].nunique()}, runs={len(run_specs_df)}"),
    integrity_row("no missing required timestamp", missing_timestamp_count == 0, f"missing_or_invalid={missing_timestamp_count}"),
    integrity_row("suitable_images == len(imgs)", ~passage_df["suitable_images_matches_imgs"], f"observations={len(passage_df)}"),
    integrity_row("prediction_start <= prediction_final", ~passage_df["prediction_order_ok"], "checked per image"),
    integrity_row("active_inferences <= 1", passage_df["active_inferences"].gt(1).fillna(True), "one prediction worker"),
    integrity_row("active_inferences + not_started_inferences == residual_inferences", (passage_df["active_inferences"] + passage_df["not_started_inferences"]).ne(passage_df["residual_inferences"]).fillna(True), f"observations={len(passage_df)}"),
    integrity_row("residual_inferences <= suitable_images", passage_df["residual_inferences"].gt(passage_df["suitable_images"]).fillna(True), f"observations={len(passage_df)}"),
    integrity_row("residual_inferences == count(prediction_final > last_capture)", passage_df["residual_inferences"].ne(passage_df["residual_by_prediction_final"]).fillna(True), f"observations={len(passage_df)}"),
    integrity_row("prediction_drain_s >= 0", passage_df["prediction_drain_s"].lt(0) | passage_df["prediction_drain_s"].isna(), f"observations={len(passage_df)}"),
    integrity_row("post_capture_latency_s >= 0", passage_df["post_capture_latency_s"].lt(0) | passage_df["post_capture_latency_s"].isna(), f"observations={len(passage_df)}"),
    integrity_row("finalization_overhead_s >= 0", passage_df["finalization_overhead_s"].lt(0) | passage_df["finalization_overhead_s"].isna(), f"observations={len(passage_df)}"),
    integrity_row(
        "post_capture_latency_s approximately equals prediction_drain_s + finalization_overhead_s",
        latency_error.gt(LATENCY_IDENTITY_ATOL_S) | latency_error.isna(),
        f"absolute tolerance={LATENCY_IDENTITY_ATOL_S:g} s; max error={latency_error.max():.3g} s",
    ),
]
integrity_df = pd.DataFrame(integrity_rows)

if not integrity_df["passed"].all():
    failures = integrity_df.loc[~integrity_df["passed"]]
    raise AssertionError("Mandatory integrity checks failed:\n" + failures.to_string(index=False))


run_passage_rows = []
for (configuration, run), group in passage_df.groupby(["configuration", "run"], sort=False, observed=True):
    residual_sum = group["residual_inferences"].sum(min_count=1)
    not_started_sum = group["not_started_inferences"].sum(min_count=1)
    run_passage_rows.append(
        {
            "configuration": configuration,
            "run": run,
            "frames_mean": float(group["total_images"].mean()),
            "suited_mean": float(group["suitable_images"].mean()),
            "latency_mean": float(group["post_capture_latency_s"].mean()),
            "latency_p95": p95(group["post_capture_latency_s"]),
            "latency_max": float(group["post_capture_latency_s"].max()),
            "residual_passage_fraction": float(group["has_residual"].mean()),
            "residual_mean": float(group["residual_inferences"].mean()),
            "residual_p95": p95(group["residual_inferences"]),
            "drain_mean": float(group["prediction_drain_s"].mean()),
            "active_mean": float(group["active_inferences"].mean()),
            "not_started_mean": float(group["not_started_inferences"].mean()),
            "not_started_share": float(not_started_sum / residual_sum) if pd.notna(residual_sum) and residual_sum != 0 else math.nan,
        }
    )
run_passage_df = pd.DataFrame(run_passage_rows)

latency_by_animal = (
    passage_df.groupby(["configuration", "animal_id"], observed=True)
    .agg(latency_mean_across_runs=("post_capture_latency_s", "mean"), n_runs=("run", "nunique"))
    .reset_index()
)
animal_counts = latency_by_animal.groupby("configuration", observed=True)["animal_id"].nunique().reindex(CONFIG_ORDER)
if not animal_counts.eq(EXPECTED_PASSAGES).all() or not latency_by_animal["n_runs"].eq(5).all():
    raise AssertionError("Latency boxplot aggregation did not yield 184 animals with five runs per configuration.")

print("Mandatory integrity checks: all passed")
display(integrity_df)
print("Fixed campaign: 8,280 passage-run observations")
print("All configurations including Original:", f"{len(passage_df):,}", "passage-run observations")

In [ ]:
def crop_series(frame: pd.DataFrame, time_column: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    result = frame.copy()
    result[time_column] = pd.to_datetime(result[time_column], errors="coerce")
    result = result.dropna(subset=[time_column]).sort_values(time_column).drop_duplicates(time_column)
    return result[(result[time_column] >= start) & (result[time_column] <= end)].copy()


def integrate_power_kj(path: Path, start: pd.Timestamp, end: pd.Timestamp) -> tuple[float, dict[str, Any]]:
    power = pd.read_csv(path, skipinitialspace=True)
    power["timestamp"] = pd.to_datetime(power["Datetime"], errors="coerce")
    power["power_w"] = pd.to_numeric(power["Power[W]"], errors="coerce")
    power = power.dropna(subset=["timestamp", "power_w"]).sort_values("timestamp").drop_duplicates("timestamp")
    if len(power) < 2 or start < power["timestamp"].min() or end > power["timestamp"].max():
        raise ValueError(f"Power coverage does not contain the operational window for {path}")
    origin = power["timestamp"].iloc[0]
    all_t = (power["timestamp"] - origin).dt.total_seconds().to_numpy(float)
    start_t = (start - origin).total_seconds()
    end_t = (end - origin).total_seconds()
    inside = power[(power["timestamp"] > start) & (power["timestamp"] < end)]
    times = np.concatenate([[start_t], (inside["timestamp"] - origin).dt.total_seconds().to_numpy(float), [end_t]])
    watts = np.concatenate([[np.interp(start_t, all_t, power["power_w"])], inside["power_w"].to_numpy(float), [np.interp(end_t, all_t, power["power_w"])]])
    energy_j = float(np.trapezoid(watts, times))
    audit = {"n_power_points": len(times), "coverage_start": power["timestamp"].min(), "coverage_end": power["timestamp"].max(), "window_s": end_t - start_t}
    return energy_j / 1000.0, audit


window_lookup = run_windows_df.set_index(["configuration", "run"])
resource_rows: list[dict[str, Any]] = []
telemetry_issues: list[dict[str, Any]] = []

for spec in run_specs_df.itertuples(index=False):
    configuration = str(spec.configuration)
    start = window_lookup.loc[(configuration, spec.run), "analysis_start"]
    end = window_lookup.loc[(configuration, spec.run), "analysis_end"]
    if pd.isna(start) or pd.isna(end) or start >= end:
        telemetry_issues.append({"configuration": configuration, "run": spec.run, "issue": "invalid_analysis_window"})
        continue

    cpu = pd.read_csv(spec.cpu_path)
    cpu = crop_series(cpu, "timestamp", start, end)
    cores = [column for column in cpu.columns if column.startswith("cpu_core_")]
    for column in cores:
        cpu[column] = pd.to_numeric(cpu[column], errors="coerce")
    cpu["cpu_utilization"] = cpu[cores].mean(axis=1)

    mem = crop_series(pd.read_csv(spec.mem_path), "timestamp", start, end)
    mem["ram_percent"] = pd.to_numeric(mem["percent"], errors="coerce")
    temp = crop_series(pd.read_csv(spec.temp_path), "timestamp", start, end)
    temp["temperature_c"] = pd.to_numeric(temp["temperature"], errors="coerce")

    for name, values in [("cpu", cpu["cpu_utilization"]), ("ram", mem["ram_percent"]), ("temperature", temp["temperature_c"])]:
        if values.dropna().empty:
            telemetry_issues.append({"configuration": configuration, "run": spec.run, "issue": f"empty_{name}_window"})

    try:
        energy_kj, energy_audit = integrate_power_kj(Path(spec.power_path), start, end)
    except Exception as exc:
        telemetry_issues.append({"configuration": configuration, "run": spec.run, "issue": "power_integration", "details": str(exc)})
        energy_kj, energy_audit = math.nan, {}

    resource_rows.append(
        {
            "configuration": configuration,
            "fps": spec.fps,
            "run": spec.run,
            "analysis_start": start,
            "analysis_end": end,
            "cpu_mean": float(cpu["cpu_utilization"].mean()),
            "cpu_p95": p95(cpu["cpu_utilization"]),
            "cpu_max": float(cpu["cpu_utilization"].max()),
            "ram_mean": float(mem["ram_percent"].mean()),
            "ram_p95": p95(mem["ram_percent"]),
            "ram_max": float(mem["ram_percent"].max()),
            "temp_mean": float(temp["temperature_c"].mean()),
            "temp_p95": p95(temp["temperature_c"]),
            "temp_max": float(temp["temperature_c"].max()),
            "energy_kj": energy_kj,
            "cpu_samples": int(cpu["cpu_utilization"].notna().sum()),
            "ram_samples": int(mem["ram_percent"].notna().sum()),
            "temp_samples": int(temp["temperature_c"].notna().sum()),
            **energy_audit,
        }
    )

resource_run_df = pd.DataFrame(resource_rows)
telemetry_issues_df = pd.DataFrame(telemetry_issues)
if len(resource_run_df) != 50 or not telemetry_issues_df.empty:
    raise AssertionError("Telemetry audit failed:\n" + telemetry_issues_df.to_string(index=False))

run_df = run_passage_df.merge(resource_run_df, on=["configuration", "run"], how="inner", validate="one_to_one")
if len(run_df) != 50:
    raise AssertionError(f"Expected 50 consolidated run rows; obtained {len(run_df)}")


def add_summary(record: dict[str, Any], name: str, values: pd.Series) -> None:
    summary = ci95(values)
    record[name] = summary["mean"]
    record[f"{name}_std"] = summary["std"]
    record[f"{name}_ci95_low"] = summary["ci95_low"]
    record[f"{name}_ci95_high"] = summary["ci95_high"]
    record[f"{name}_n_runs"] = summary["n"]


source_rows = []
for configuration in CONFIG_ORDER:
    group = run_df[run_df["configuration"].eq(configuration)]
    passage_group = passage_df[passage_df["configuration"].eq(configuration)]
    record: dict[str, Any] = {"input": configuration, "configured_fps": float(configuration) if configuration != "Original" else math.nan}
    mappings = {
        "frames_per_passage_mean_of_run_means": "frames_mean",
        "suited_per_passage_mean_of_run_means": "suited_mean",
        "cpu_mean_of_run_means": "cpu_mean",
        "cpu_mean_of_run_p95": "cpu_p95",
        "cpu_mean_of_run_maxima": "cpu_max",
        "ram_mean_of_run_means": "ram_mean",
        "ram_mean_of_run_p95": "ram_p95",
        "ram_mean_of_run_maxima": "ram_max",
        "temperature_mean_of_run_means": "temp_mean",
        "temperature_mean_of_run_p95": "temp_p95",
        "temperature_mean_of_run_maxima": "temp_max",
        "energy_mean_kj": "energy_kj",
        "latency_mean_of_run_means_s": "latency_mean",
        "latency_mean_of_run_p95_s": "latency_p95",
        "latency_mean_of_run_maxima_s": "latency_max",
        "residual_passages_mean_fraction": "residual_passage_fraction",
        "residual_inferences_mean_of_run_means": "residual_mean",
        "residual_inferences_mean_of_run_p95": "residual_p95",
        "drain_mean_of_run_means_s": "drain_mean",
        "not_started_share_mean_fraction": "not_started_share",
        "active_inferences_mean_of_run_means": "active_mean",
        "not_started_inferences_mean_of_run_means": "not_started_mean",
    }
    for output_name, run_column in mappings.items():
        add_summary(record, output_name, group[run_column])
    record["cpu_absolute_peak_across_all_runs"] = float(group["cpu_max"].max())
    record["ram_absolute_peak_across_all_runs"] = float(group["ram_max"].max())
    record["temperature_absolute_peak_across_all_runs_c"] = float(group["temp_max"].max())
    record["latency_absolute_peak_across_all_passages_and_runs_s"] = float(passage_group["post_capture_latency_s"].max())
    record["passage_run_observations"] = len(passage_group)
    source_rows.append(record)

source_tables_df = pd.DataFrame(source_rows)
source_tables_df["input"] = pd.Categorical(source_tables_df["input"], CONFIG_ORDER, ordered=True)
source_tables_df = source_tables_df.sort_values("input").reset_index(drop=True)
source_tables_df["input"] = source_tables_df["input"].astype(str)
source_tables_df.to_csv(TABLE_DIR / "source_tables.csv", index=False, float_format="%.12g")

print("Run-level table shape:", run_df.shape)
print("Source table shape:", source_tables_df.shape)
display(source_tables_df[["input", "frames_per_passage_mean_of_run_means", "cpu_mean_of_run_means", "energy_mean_kj", "latency_mean_of_run_means_s", "residual_passages_mean_fraction"]].round(6))

## Revised IEEE presentation artifacts

The cells below only revise presentation and layout. They reuse the audited raw-data calculations above without changing metric definitions, run-level aggregation, or integrity checks. Fixed-FPS configurations are plotted at equally spaced categorical positions.

In [ ]:
# Compact full-width IEEE operational table.
from pathlib import Path
import shutil
import subprocess
import tempfile

TARGET_WIDTH_IN = 7.16
TABLE_FONT_SIZE = r"\scriptsize"
TABLE_TABCOLSEP_PT = 1.6
TABLE_ARRAYSTRETCH = 1.02
PREVIOUS_ARRAYSTRETCH = 1.07
EXPECTED_ORDER = ["Original", "1", "2", "3", "4", "5", "10", "15", "20", "30"]


def find_compact_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    if "REPO_ROOT" in globals():
        candidates.insert(0, Path(REPO_ROOT).resolve())
    for candidate in candidates:
        baseline = (
            candidate
            / "article_artifacts_compact"
            / "validation"
            / "operational_numeric_baseline.csv"
        )
        if baseline.is_file():
            return candidate
    raise FileNotFoundError("Could not locate operational_numeric_baseline.csv")


COMPACT_REPO_ROOT = find_compact_root(Path.cwd())
COMPACT_ROOT = COMPACT_REPO_ROOT / "article_artifacts_compact"
COMPACT_TABLE_DIR = COMPACT_ROOT / "tables"
COMPACT_RENDER_DIR = COMPACT_ROOT / "render_comparison"
BASELINE_PATH = COMPACT_ROOT / "validation" / "operational_numeric_baseline.csv"
COMPACT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
COMPACT_RENDER_DIR.mkdir(parents=True, exist_ok=True)

FULL_TABLE_PATH = COMPACT_TABLE_DIR / "tab_operational_results_full_compact.tex"
TEST_PDF_PATH = (
    COMPACT_RENDER_DIR / "tab_operational_results_full_compact_test.pdf"
)

VISUAL_COLUMNS = [
    "input",
    "frames_per_passage_mean_of_run_means",
    "suited_per_passage_mean_of_run_means",
    "cpu_mean_of_run_means",
    "cpu_mean_of_run_p95",
    "cpu_absolute_peak_across_all_runs",
    "ram_mean_of_run_means",
    "ram_mean_of_run_p95",
    "temperature_mean_of_run_means",
    "temperature_absolute_peak_across_all_runs_c",
    "energy_mean_kj",
    "latency_mean_of_run_means_s",
    "latency_mean_of_run_p95_s",
    "latency_absolute_peak_across_all_passages_and_runs_s",
    "residual_passages_mean_fraction",
    "residual_inferences_mean_of_run_means",
    "residual_inferences_mean_of_run_p95",
    "drain_mean_of_run_means_s",
]
NUMERIC_COLUMNS = VISUAL_COLUMNS[1:]

# Preserve the unrounded numerical layer. Formatting is applied only while
# constructing the presentation strings below.
table_numeric_values = source_tables_df.loc[:, VISUAL_COLUMNS].copy()
operational_baseline = pd.read_csv(BASELINE_PATH, float_precision="round_trip")

assert len(VISUAL_COLUMNS) == 18
assert table_numeric_values.shape == (10, 18)
assert table_numeric_values["input"].tolist() == EXPECTED_ORDER
assert operational_baseline["configuration"].tolist() == EXPECTED_ORDER
assert "Original" in table_numeric_values["input"].tolist()
assert "temperature_mean_of_run_p95" not in VISUAL_COLUMNS
assert np.isfinite(table_numeric_values[NUMERIC_COLUMNS].to_numpy(dtype=float)).all()

for column in NUMERIC_COLUMNS:
    np.testing.assert_array_equal(
        table_numeric_values[column].to_numpy(dtype=float),
        operational_baseline[column].to_numpy(dtype=float),
        err_msg=f"Baseline mismatch in {column}",
    )

np.testing.assert_array_equal(
    table_numeric_values[
        "suited_per_passage_mean_of_run_means"
    ].to_numpy(dtype=float),
    operational_baseline[
        "accepted_or_suited_per_passage_mean_of_run_means"
    ].to_numpy(dtype=float),
    err_msg="Accepted presentation values differ from the baseline alias",
)


def fmt_fixed(value: float, digits: int) -> str:
    return f"{float(value):.{digits}f}"


def fmt_latency(value: float) -> str:
    value = float(value)
    return f"{value:.3f}" if abs(value) < 10.0 else f"{value:.2f}"


latex_rows = []
for row in table_numeric_values.to_dict(orient="records"):
    latex_rows.append(
        " & ".join(
            [
                str(row["input"]),
                fmt_fixed(row["frames_per_passage_mean_of_run_means"], 2),
                fmt_fixed(row["suited_per_passage_mean_of_run_means"], 2),
                fmt_fixed(row["cpu_mean_of_run_means"], 2),
                fmt_fixed(row["cpu_mean_of_run_p95"], 2),
                fmt_fixed(row["cpu_absolute_peak_across_all_runs"], 2),
                fmt_fixed(row["ram_mean_of_run_means"], 2),
                fmt_fixed(row["ram_mean_of_run_p95"], 2),
                fmt_fixed(row["temperature_mean_of_run_means"], 2),
                fmt_fixed(row["temperature_absolute_peak_across_all_runs_c"], 2),
                fmt_fixed(row["energy_mean_kj"], 2),
                fmt_latency(row["latency_mean_of_run_means_s"]),
                fmt_latency(row["latency_mean_of_run_p95_s"]),
                fmt_latency(
                    row["latency_absolute_peak_across_all_passages_and_runs_s"]
                ),
                fmt_fixed(100.0 * row["residual_passages_mean_fraction"], 1),
                fmt_fixed(row["residual_inferences_mean_of_run_means"], 2),
                fmt_fixed(row["residual_inferences_mean_of_run_p95"], 2),
                fmt_latency(row["drain_mean_of_run_means_s"]),
            ]
        )
        + r" \\"
    )

body_lines = [latex_rows[0], r"\midrule", *latex_rows[1:]]
table_tex = "\n".join(
    [
        r"\begin{table*}[t]",
        r"\caption{Operational results for the original trace and fixed-FPS workloads.}",
        r"\label{tab:operational-results}",
        r"\centering",
        TABLE_FONT_SIZE,
        rf"\setlength{{\tabcolsep}}{{{TABLE_TABCOLSEP_PT:.1f}pt}}",
        rf"\renewcommand{{\arraystretch}}{{{TABLE_ARRAYSTRETCH:.2f}}}",
        r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}lrrrrrrrrrrrrrrrrr@{}}",
        r"\toprule",
        (
            r"\multicolumn{1}{c}{Input} & "
            r"\multicolumn{2}{c}{Workload} & "
            r"\multicolumn{3}{c}{CPU (\%)} & "
            r"\multicolumn{2}{c}{RAM (\%)} & "
            r"\multicolumn{2}{c}{Temperature ($^\circ$C)} & "
            r"\multicolumn{1}{c}{Energy (kJ)} & "
            r"\multicolumn{3}{c}{Latency (s)} & "
            r"\multicolumn{4}{c}{Residual workload} \\"
        ),
        (
            r"\cmidrule(lr){2-3}\cmidrule(lr){4-6}\cmidrule(lr){7-8}"
            r"\cmidrule(lr){9-10}\cmidrule(lr){11-11}"
            r"\cmidrule(lr){12-14}\cmidrule(l){15-18}"
        ),
        (
            r" & Frames & Accepted & Mean & P95 & Peak & Mean & P95 & "
            r"Mean & Peak & Mean & Mean & P95 & Peak & "
            r"Pass. (\%) & Inf. Mean & Inf. P95 & Drain (s) \\"
        ),
        r"\midrule",
        *body_lines,
        r"\bottomrule",
        r"\end{tabular*}",
        r"\vspace{1pt}",
        r"\begin{minipage}{\textwidth}",
        r"\scriptsize",
        r"\setlength{\parindent}{0pt}",
        r"\setlength{\parskip}{0pt}",
        (
            r"\textit{Note:} Frames and Accepted are means per passage; "
            r"Accepted denotes frames forwarded after selection. Mean and P95 "
            r"are averages of the corresponding statistics computed "
            r"independently within each run. Peak is the largest observed value "
            r"across the five runs. Original denotes the irregular native trace "
            r"and is not a fixed-FPS level. Residual workload includes active "
            r"and not-started inferences at the end of capture. Pass. denotes "
            r"passages; Inf. denotes inferences. Energy is total per run for "
            r"the same 184 passages."
        ),
        r"\end{minipage}",
        r"\end{table*}",
        "",
    ]
)

assert table_tex.count(r" \\") == 12  # two header rows plus ten data rows
assert r"\resizebox" not in table_tex
assert "adjustbox" not in table_tex
assert "landscape" not in table_tex
assert "Suited" not in table_tex
assert "Accepted" in table_tex
assert "Temperature P95" not in table_tex
assert r"\begin{tabular*}{\textwidth}" in table_tex

FULL_TABLE_PATH.write_text(table_tex, encoding="utf-8")

pdflatex_path = shutil.which("pdflatex")
latex_test_result = {
    "compiler": pdflatex_path,
    "compiled": False,
    "pdf_path": None,
    "overfull_boxes": [],
}
if pdflatex_path:
    with tempfile.TemporaryDirectory(prefix="operational_table_test_") as temp_dir:
        temp_dir = Path(temp_dir)
        test_source = temp_dir / "test.tex"
        test_source.write_text(
            "\n".join(
                [
                    r"\documentclass[conference]{IEEEtran}",
                    r"\usepackage{booktabs}",
                    r"\begin{document}",
                    r"\input{" + FULL_TABLE_PATH.as_posix() + "}",
                    r"\clearpage",
                    r"\end{document}",
                    "",
                ]
            ),
            encoding="utf-8",
        )
        completed = subprocess.run(
            [
                pdflatex_path,
                "-interaction=nonstopmode",
                "-halt-on-error",
                "-output-directory",
                str(temp_dir),
                str(test_source),
            ],
            cwd=COMPACT_REPO_ROOT,
            text=True,
            capture_output=True,
            check=False,
        )
        log_text = completed.stdout + "\n" + completed.stderr
        overfull = [
            line.strip()
            for line in log_text.splitlines()
            if "Overfull" in line
        ]
        generated_pdf = temp_dir / "test.pdf"
        assert completed.returncode == 0, log_text[-4000:]
        assert generated_pdf.is_file()
        shutil.copy2(generated_pdf, TEST_PDF_PATH)
        latex_test_result.update(
            {
                "compiled": True,
                "pdf_path": str(TEST_PDF_PATH),
                "overfull_boxes": overfull,
            }
        )

assertion_results = {
    "rows": len(table_numeric_values),
    "row_order": table_numeric_values["input"].tolist(),
    "visual_columns": len(VISUAL_COLUMNS),
    "original_present": "Original" in table_numeric_values["input"].tolist(),
    "all_underlying_values_exactly_match_baseline": True,
    "rounding_before_presentation": False,
    "temperature_p95_absent": True,
}
estimated_row_spacing_reduction_percent = (
    100.0 * (PREVIOUS_ARRAYSTRETCH - TABLE_ARRAYSTRETCH) / PREVIOUS_ARRAYSTRETCH
)

print("Wrote:", FULL_TABLE_PATH)
print("Target width (in):", TARGET_WIDTH_IN)
print("Font:", TABLE_FONT_SIZE)
print("tabcolsep (pt):", TABLE_TABCOLSEP_PT)
print("arraystretch:", TABLE_ARRAYSTRETCH)
print(
    "Estimated row-spacing reduction versus arraystretch 1.07 (%):",
    round(estimated_row_spacing_reduction_percent, 3),
)
print("Assertions:", assertion_results)
print("LaTeX test:", latex_test_result)
display(table_numeric_values)


In [ ]:
# Compact alternative A: CPU utilization and temperature on the real FPS axis.
COMPACT_FIGURE_DIR = REPO_ROOT / "article_artifacts_compact" / "figures"
COMPACT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_BASELINE_PATH = (
    REPO_ROOT
    / "article_artifacts_compact"
    / "validation"
    / "operational_plot_arrays_baseline.npz"
)
PLOT_BASELINE = np.load(PLOT_BASELINE_PATH)
COMPACT_FPS = np.asarray(FIXED_FPS, dtype=int)


def compact_metric_summary(metric: str) -> dict[str, np.ndarray]:
    summaries = []
    for fps in FIXED_FPS:
        values = run_df.loc[run_df["configuration"].eq(str(fps)), metric]
        summaries.append(ci95(values))
    return {
        "fps": COMPACT_FPS.copy(),
        "mean": np.asarray([item["mean"] for item in summaries], dtype=float),
        "ci95_low": np.asarray(
            [item["ci95_low"] for item in summaries], dtype=float
        ),
        "ci95_high": np.asarray(
            [item["ci95_high"] for item in summaries], dtype=float
        ),
    }


def compact_yerr(summary: dict[str, np.ndarray]) -> np.ndarray:
    return np.vstack(
        [
            summary["mean"] - summary["ci95_low"],
            summary["ci95_high"] - summary["mean"],
        ]
    )


def assert_summary_matches_baseline(
    summary: dict[str, np.ndarray],
    baseline_prefix: str,
) -> None:
    np.testing.assert_array_equal(
        summary["fps"],
        PLOT_BASELINE["fps_levels"],
        err_msg=f"FPS mismatch for {baseline_prefix}",
    )
    for field in ("mean", "ci95_low", "ci95_high"):
        baseline_key = (
            baseline_prefix if field == "mean" else f"{baseline_prefix}_{field}"
        )
        np.testing.assert_array_equal(
            summary[field],
            PLOT_BASELINE[baseline_key],
            err_msg=f"Mismatch in {baseline_key}",
        )
    yerr = compact_yerr(summary)
    np.testing.assert_array_equal(
        yerr[0],
        PLOT_BASELINE[f"{baseline_prefix}_yerr_lower"],
        err_msg=f"Lower yerr mismatch for {baseline_prefix}",
    )
    np.testing.assert_array_equal(
        yerr[1],
        PLOT_BASELINE[f"{baseline_prefix}_yerr_upper"],
        err_msg=f"Upper yerr mismatch for {baseline_prefix}",
    )


def compact_errorbar(
    ax: mpl.axes.Axes,
    summary: dict[str, np.ndarray],
    style_name: str,
    label: str,
) -> None:
    ax.errorbar(
        summary["fps"],
        summary["mean"],
        yerr=compact_yerr(summary),
        linewidth=1.25,
        markersize=4.2,
        capsize=2.6,
        capthick=0.8,
        elinewidth=0.9,
        label=label,
        zorder=3,
        **STYLES[style_name],
    )


def compact_real_fps_axis(ax: mpl.axes.Axes) -> None:
    ax.set_xticks(COMPACT_FPS, [str(value) for value in COMPACT_FPS])
    ax.set_xlim(0, 31)
    ax.tick_params(axis="both", labelsize=7.5, pad=1.5)
    ax.grid(axis="y", linewidth=0.55)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def compact_panel_label(ax: mpl.axes.Axes, label: str) -> None:
    ax.text(
        0.0,
        1.015,
        label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=8.5,
        fontweight="bold",
    )


def compact_temperature_background(ax: mpl.axes.Axes) -> None:
    ax.axhspan(80, 85, color="#E8E8E8", zorder=0)
    ax.axhspan(85, 88, color="#CFCFCF", zorder=0)
    ax.axhline(80, color="#666666", linestyle="--", linewidth=0.8, zorder=1)
    ax.axhline(85, color="#555555", linestyle=":", linewidth=0.8, zorder=1)
    ax.text(
        30.4,
        82.5,
        "80-85 °C",
        ha="right",
        va="center",
        fontsize=7.0,
        color="#444444",
    )


def save_compact_figure(
    fig: mpl.figure.Figure,
    stem: str,
    expected_size: tuple[float, float],
) -> tuple[Path, Path]:
    np.testing.assert_allclose(
        fig.get_size_inches(),
        np.asarray(expected_size, dtype=float),
        rtol=0,
        atol=1e-12,
    )
    pdf_path = COMPACT_FIGURE_DIR / f"{stem}.pdf"
    png_path = COMPACT_FIGURE_DIR / f"{stem}.png"
    fig.savefig(
        pdf_path,
        format="pdf",
        dpi=300,
        facecolor="white",
        bbox_inches=None,
        pad_inches=0,
    )
    fig.savefig(
        png_path,
        format="png",
        dpi=300,
        facecolor="white",
        bbox_inches=None,
        pad_inches=0,
    )
    display(fig)
    plt.close(fig)
    return pdf_path, png_path


cpu_mean_arrays = compact_metric_summary("cpu_mean")
cpu_p95_arrays = compact_metric_summary("cpu_p95")
temperature_mean_arrays = compact_metric_summary("temp_mean")
temperature_mean_run_maximum_arrays = compact_metric_summary("temp_max")

assert_summary_matches_baseline(cpu_mean_arrays, "cpu_mean")
assert_summary_matches_baseline(cpu_p95_arrays, "cpu_p95")
assert_summary_matches_baseline(temperature_mean_arrays, "temperature_mean")
assert_summary_matches_baseline(
    temperature_mean_run_maximum_arrays,
    "temperature_mean_run_maximum",
)
assert mpl.rcParams["pdf.fonttype"] == 42

FIGURE_A_SIZE_IN = (7.16, 2.25)
fig_a, axes_a = plt.subplots(1, 2, figsize=FIGURE_A_SIZE_IN)

ax_cpu_a, ax_temp_a = axes_a
compact_errorbar(ax_cpu_a, cpu_mean_arrays, "mean", "Mean")
compact_errorbar(ax_cpu_a, cpu_p95_arrays, "p95", "P95")
compact_real_fps_axis(ax_cpu_a)
ax_cpu_a.set_ylim(0, 100)
ax_cpu_a.set_yticks(np.arange(0, 101, 20))
ax_cpu_a.set_ylabel("CPU utilization (%)", fontsize=8.5, labelpad=2.0)
ax_cpu_a.legend(
    loc="upper left",
    ncol=2,
    frameon=False,
    fontsize=7.5,
    handlelength=1.8,
    columnspacing=0.9,
    borderaxespad=0.25,
)
compact_panel_label(ax_cpu_a, "(a)")

compact_temperature_background(ax_temp_a)
compact_errorbar(ax_temp_a, temperature_mean_arrays, "mean", "Mean")
compact_errorbar(
    ax_temp_a,
    temperature_mean_run_maximum_arrays,
    "maximum",
    "Mean run maximum",
)
compact_real_fps_axis(ax_temp_a)
ax_temp_a.set_ylim(54, 88)
ax_temp_a.set_yticks([55, 60, 65, 70, 75, 80, 85])
ax_temp_a.set_ylabel("Temperature (°C)", fontsize=8.5, labelpad=2.0)
ax_temp_a.legend(
    loc="upper left",
    ncol=1,
    frameon=False,
    fontsize=7.3,
    handlelength=1.8,
    borderaxespad=0.25,
    labelspacing=0.25,
)
compact_panel_label(ax_temp_a, "(b)")

fig_a.supxlabel("Configured FPS", fontsize=8.5, y=0.025)
fig_a.subplots_adjust(
    left=0.070,
    right=0.990,
    bottom=0.225,
    top=0.905,
    wspace=0.225,
)
figure_a_paths = save_compact_figure(
    fig_a,
    "fig_cpu_temperature_compact_A",
    FIGURE_A_SIZE_IN,
)

figure_a_assertions = {
    "fps_levels": COMPACT_FPS.tolist(),
    "cpu_mean_and_ci_exact": True,
    "cpu_p95_and_ci_exact": True,
    "temperature_mean_and_ci_exact": True,
    "temperature_mean_run_maximum_and_ci_exact": True,
    "temperature_upper_curve_definition": (
        "mean of the five within-run temporal maxima"
    ),
    "pdf_fonttype": int(mpl.rcParams["pdf.fonttype"]),
}
print("Alternative A:", figure_a_paths, FIGURE_A_SIZE_IN)
print("Alternative A assertions:", figure_a_assertions)


In [ ]:
# Compact alternative B: CPU, temperature, and total energy per run.
energy_mean_arrays = compact_metric_summary("energy_kj")
assert_summary_matches_baseline(energy_mean_arrays, "energy_mean_kj")
np.testing.assert_array_equal(
    energy_mean_arrays["fps"],
    PLOT_BASELINE["fps_levels"],
)

FIGURE_B_SIZE_IN = (7.16, 2.35)
fig_b, axes_b = plt.subplots(1, 3, figsize=FIGURE_B_SIZE_IN)
ax_cpu_b, ax_temp_b, ax_energy_b = axes_b

compact_errorbar(ax_cpu_b, cpu_mean_arrays, "mean", "Mean")
compact_errorbar(ax_cpu_b, cpu_p95_arrays, "p95", "P95")
compact_real_fps_axis(ax_cpu_b)
ax_cpu_b.set_ylim(0, 100)
ax_cpu_b.set_yticks(np.arange(0, 101, 20))
ax_cpu_b.set_ylabel("CPU utilization (%)", fontsize=8.2, labelpad=1.5)
ax_cpu_b.legend(
    loc="upper left",
    ncol=2,
    frameon=False,
    fontsize=7.2,
    handlelength=1.65,
    columnspacing=0.7,
    borderaxespad=0.2,
)
compact_panel_label(ax_cpu_b, "(a)")

compact_temperature_background(ax_temp_b)
compact_errorbar(ax_temp_b, temperature_mean_arrays, "mean", "Mean")
compact_errorbar(
    ax_temp_b,
    temperature_mean_run_maximum_arrays,
    "maximum",
    "Mean run maximum",
)
compact_real_fps_axis(ax_temp_b)
ax_temp_b.set_ylim(54, 88)
ax_temp_b.set_yticks([55, 60, 65, 70, 75, 80, 85])
ax_temp_b.set_ylabel("Temperature (°C)", fontsize=8.2, labelpad=1.5)
ax_temp_b.legend(
    loc="lower right",
    ncol=1,
    frameon=False,
    fontsize=7.0,
    handlelength=1.6,
    borderaxespad=0.2,
    labelspacing=0.2,
)
compact_panel_label(ax_temp_b, "(b)")

ax_energy_b.errorbar(
    energy_mean_arrays["fps"],
    energy_mean_arrays["mean"],
    yerr=compact_yerr(energy_mean_arrays),
    color=COLORS["mean"],
    linestyle="-",
    marker="o",
    ecolor="#222222",
    linewidth=1.25,
    markersize=4.0,
    capsize=3.0,
    capthick=0.9,
    elinewidth=1.0,
    label="Mean (5 runs)",
    zorder=3,
)
compact_real_fps_axis(ax_energy_b)
ax_energy_b.set_ylim(4.0, 12.2)
ax_energy_b.set_yticks([4, 6, 8, 10, 12])
ax_energy_b.set_ylabel("Total energy per run (kJ)", fontsize=8.2, labelpad=1.5)
ax_energy_b.legend(
    loc="upper left",
    frameon=False,
    fontsize=7.2,
    handlelength=1.65,
    borderaxespad=0.2,
)
compact_panel_label(ax_energy_b, "(c)")

for axis in axes_b:
    axis.tick_params(
        axis="x", labelsize=7.0, pad=1.0, labelrotation=45
    )
    for tick_label in axis.get_xticklabels():
        tick_label.set_horizontalalignment("right")

fig_b.supxlabel("Configured FPS", fontsize=8.5, y=0.025)
fig_b.subplots_adjust(
    left=0.057,
    right=0.995,
    bottom=0.220,
    top=0.910,
    wspace=0.340,
)
figure_b_paths = save_compact_figure(
    fig_b,
    "fig_cpu_temperature_energy_compact_B",
    FIGURE_B_SIZE_IN,
)

figure_b_assertions = {
    "fps_levels": energy_mean_arrays["fps"].tolist(),
    "energy_mean_and_ci_exact": True,
    "energy_definition": "total per run for the same 184 passages",
    "normalization": None,
    "power_metric": None,
    "pdf_fonttype": int(mpl.rcParams["pdf.fonttype"]),
}
print("Alternative B:", figure_b_paths, FIGURE_B_SIZE_IN)
print("Alternative B assertions:", figure_b_assertions)


In [ ]:
# Compact absolute linear boxplot: 184 passage means per configured FPS.
COMPACT_LATENCY_FIGURE_DIR = (
    REPO_ROOT / "article_artifacts_compact" / "figures"
)
COMPACT_LATENCY_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
LATENCY_BASELINE_PATH = (
    REPO_ROOT
    / "article_artifacts_compact"
    / "validation"
    / "latency_distributions_baseline.npz"
)
latency_baseline = np.load(LATENCY_BASELINE_PATH)

fixed_latency_data = [
    latency_by_animal.loc[
        latency_by_animal["configuration"].eq(str(fps)),
        "latency_mean_across_runs",
    ].to_numpy(float)
    for fps in FIXED_FPS
]
latency_observations = np.vstack(fixed_latency_data)
latency_box_means = latency_observations.mean(axis=1)
latency_box_maxima = latency_observations.max(axis=1)

assert latency_observations.shape == (9, EXPECTED_PASSAGES)
np.testing.assert_array_equal(
    np.asarray(FIXED_FPS, dtype=int),
    latency_baseline["fps_levels"],
)
np.testing.assert_array_equal(
    latency_observations,
    latency_baseline["latency_observations_s"],
)
np.testing.assert_array_equal(
    latency_box_means,
    latency_baseline["box_mean_s"],
)
np.testing.assert_array_equal(
    latency_box_maxima,
    latency_baseline["latency_observations_s"].max(axis=1),
)
np.testing.assert_array_equal(
    np.full(9, EXPECTED_PASSAGES, dtype=int),
    latency_baseline["n_observations"],
)
assert np.isfinite(latency_observations).all()
assert float(latency_box_maxima.max()) < 110.0
assert mpl.rcParams["pdf.fonttype"] == 42


def latency_outliers_1_5_iqr(values: np.ndarray) -> np.ndarray:
    q1, q3 = np.percentile(values, [25, 75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return values[(values < lower) | (values > upper)]


expected_latency_outliers = [
    latency_outliers_1_5_iqr(values) for values in fixed_latency_data
]
latency_positions = np.arange(1, len(FIXED_FPS) + 1, dtype=float)
LATENCY_FIGURE_SIZE_IN = (3.50, 1.85)

fig_latency, ax_latency = plt.subplots(figsize=LATENCY_FIGURE_SIZE_IN)
latency_box_artists = ax_latency.boxplot(
    fixed_latency_data,
    positions=latency_positions,
    widths=0.58,
    whis=1.5,
    patch_artist=True,
    showfliers=True,
    boxprops={
        "facecolor": COLORS["box_fill"],
        "edgecolor": COLORS["box_edge"],
        "linewidth": 0.95,
    },
    whiskerprops={"color": COLORS["box_edge"], "linewidth": 0.85},
    capprops={"color": COLORS["box_edge"], "linewidth": 0.85},
    medianprops={"color": COLORS["median"], "linewidth": 1.05},
    flierprops={
        "marker": "o",
        "markersize": 2.0,
        "markerfacecolor": "none",
        "markeredgecolor": COLORS["outliers"],
        "markeredgewidth": 0.50,
        "linestyle": "none",
    },
)

assert len(latency_box_artists["boxes"]) == 9
assert len(latency_box_artists["fliers"]) == 9
observed_outlier_counts = []
for expected, artist in zip(
    expected_latency_outliers,
    latency_box_artists["fliers"],
):
    observed = np.asarray(artist.get_ydata(), dtype=float)
    np.testing.assert_array_equal(np.sort(observed), np.sort(expected))
    observed_outlier_counts.append(int(len(observed)))

ax_latency.plot(
    latency_positions,
    latency_box_means,
    linestyle="none",
    marker="o",
    markersize=4.2,
    color=COLORS["mean"],
    zorder=4,
)
ax_latency.set_xticks(
    latency_positions,
    [str(value) for value in FIXED_FPS],
)
ax_latency.set_xlim(0.55, 9.35)
ax_latency.set_ylim(0, 100)
ax_latency.set_yticks([0, 20, 40, 60, 80, 100])
ax_latency.set_xlabel(
    "Configured FPS",
    fontsize=8.5,
    labelpad=2.0,
)
ax_latency.set_ylabel(
    "Post-capture latency (s)",
    fontsize=8.5,
    labelpad=2.0,
)
ax_latency.tick_params(axis="x", labelsize=7.5, pad=1.5)
ax_latency.tick_params(axis="y", labelsize=7.5, pad=1.5)
ax_latency.grid(axis="y", linewidth=0.55)
ax_latency.spines["top"].set_visible(False)
ax_latency.spines["right"].set_visible(False)
assert ax_latency.get_yscale() == "linear"

fig_latency.subplots_adjust(
    left=0.185,
    right=0.985,
    bottom=0.245,
    top=0.995,
)
np.testing.assert_allclose(
    fig_latency.get_size_inches(),
    np.asarray(LATENCY_FIGURE_SIZE_IN),
    rtol=0,
    atol=1e-12,
)

latency_pdf_path = (
    COMPACT_LATENCY_FIGURE_DIR
    / "fig_latency_box_absolute_single_compact.pdf"
)
latency_png_path = (
    COMPACT_LATENCY_FIGURE_DIR
    / "fig_latency_box_absolute_single_compact.png"
)
fig_latency.savefig(
    latency_pdf_path,
    format="pdf",
    dpi=300,
    facecolor="white",
    bbox_inches="tight",
    pad_inches=0.01,
)
fig_latency.savefig(
    latency_png_path,
    format="png",
    dpi=300,
    facecolor="white",
    bbox_inches="tight",
    pad_inches=0.01,
)
display(fig_latency)
plt.close(fig_latency)

latency_compact_assertions = {
    "boxes": len(fixed_latency_data),
    "observations_per_box": [
        int(len(values)) for values in fixed_latency_data
    ],
    "distributions_exact": True,
    "means_exact": True,
    "maxima_exact": True,
    "whiskers": "1.5 x IQR",
    "showfliers": True,
    "outlier_counts": observed_outlier_counts,
    "outliers_total": int(sum(observed_outlier_counts)),
    "y_scale": "linear",
    "y_limits_s": [0, 100],
    "pdf_fonttype": int(mpl.rcParams["pdf.fonttype"]),
}
print(
    "Compact latency figure:",
    latency_pdf_path,
    latency_png_path,
    LATENCY_FIGURE_SIZE_IN,
)
print("Latency assertions:", latency_compact_assertions)


In [ ]:

captions = r'''% Candidate captions for the revised article artifacts.
\newcommand{\captionCpuTemperature}{CPU utilization and operating temperature across the nine fixed-FPS configurations. Points show the mean of the corresponding run-level statistics and error bars show 95\% Student-$t$ confidence intervals over five complete runs. The light-gray thermal region begins at the control-onset threshold of 80\,$^{\circ}$C and extends to the top of the panel; a darker-gray overlay marks temperatures above 85\,$^{\circ}$C, representing a more restrictive thermal condition. Mean run maximum is the mean of the five within-run maxima, not the absolute campaign peak.}
\newcommand{\captionEnergyPoint}{Mean energy per run across fixed-FPS configurations, with 95\% Student-$t$ confidence intervals over five runs. Configurations are equally spaced categorical levels and energy is obtained by trapezoidal integration using the recorded timestamps.}
\newcommand{\captionEnergyBar}{Mean energy per run across fixed-FPS configurations, shown as bars with 95\% Student-$t$ confidence intervals over five runs. Configurations are equally spaced categorical levels and energy is obtained by trapezoidal integration using the recorded timestamps.}
\newcommand{\captionRealX}{The same CPU, temperature, or energy summaries with configured FPS represented on its real numeric x-axis; error bars are 95\% Student-$t$ confidence intervals over five complete runs.}
\newcommand{\captionEnergyPointRealXFilled}{Mean energy per run on the real numeric FPS axis, with the area below the mean curve filled for visual emphasis and dark error bars showing 95\% Student-$t$ confidence intervals over five runs.}
\newcommand{\captionLatencyAbsolute}{Post-capture latency on a linear absolute scale. Each box summarizes 184 passage-level means, with each passage mean calculated over five runs; boxes show the interquartile range, center lines the median, whiskers 1.5 times the interquartile range, and blue circles the mean. Outliers are displayed and were not removed.}
\newcommand{\captionLatencyInset}{Post-capture latency on a linear absolute 0--110\,s scale. Each box summarizes 184 passage-level means, each calculated over five runs, and all outliers remain visible in the main panel. The inset enlarges the 1--15 FPS regime on a separate linear scale without changing the absolute main-panel scale.}
'''
(REVISED_ROOT / "article_figure_captions_revised.tex").write_text(captions, encoding="utf-8")


def pdf_size_inches(path: Path) -> tuple[float, float]:
    match = re.search(rb"/MediaBox\s*\[\s*[-0-9.]+\s+[-0-9.]+\s+([-0-9.]+)\s+([-0-9.]+)\s*\]", path.read_bytes())
    if not match:
        return (math.nan, math.nan)
    return (float(match.group(1)) / 72.0, float(match.group(2)) / 72.0)


figure_dimensions = {
    stem: pdf_size_inches(REVISED_FIGURE_DIR / f"{stem}.pdf")
    for stem in FIGURE_SIZES_IN
}

pdflatex_path = shutil.which("pdflatex")
latex_audit = "`pdflatex` was not available; IEEEtran compilation and fit were not verified, and this audit makes no claim that the tables fit naturally."
if pdflatex_path:
    test_dir = Path(tempfile.mkdtemp(prefix="ieee_revised_"))
    document = rf'''\documentclass[conference]{{IEEEtran}}
\usepackage{{booktabs,threeparttable,graphicx}}
\begin{{document}}
\input{{{(REVISED_TABLE_DIR / 'tab_operational_results_full_revised.tex').as_posix()}}}
\input{{{(REVISED_TABLE_DIR / 'tab_workload_resources.tex').as_posix()}}}
\input{{{(REVISED_TABLE_DIR / 'tab_residual_workload_compact.tex').as_posix()}}}
\begin{{figure*}}\centering\includegraphics[width=\textwidth]{{{(REVISED_FIGURE_DIR / 'fig_cpu_temperature.pdf').as_posix()}}}\end{{figure*}}
\begin{{figure}}\centering\includegraphics[width=\columnwidth]{{{(REVISED_FIGURE_DIR / 'fig_energy_point.pdf').as_posix()}}}\end{{figure}}
\begin{{figure}}\centering\includegraphics[width=\columnwidth]{{{(REVISED_FIGURE_DIR / 'fig_energy_bar.pdf').as_posix()}}}\end{{figure}}
\begin{{figure}}\centering\includegraphics[width=\columnwidth]{{{(REVISED_FIGURE_DIR / 'fig_latency_box_absolute_single.pdf').as_posix()}}}\end{{figure}}
\begin{{figure}}\centering\includegraphics[width=\columnwidth]{{{(REVISED_FIGURE_DIR / 'fig_latency_box_absolute_inset_single.pdf').as_posix()}}}\end{{figure}}
\end{{document}}
'''
    tex_path = test_dir / "revised_test.tex"
    tex_path.write_text(document, encoding="utf-8")
    result = subprocess.run([pdflatex_path, "-interaction=nonstopmode", tex_path.name], cwd=test_dir, capture_output=True, text=True)
    log_text = (test_dir / "revised_test.log").read_text(errors="replace") if (test_dir / "revised_test.log").exists() else result.stdout
    overfull = [line.strip() for line in log_text.splitlines() if "Overfull" in line]
    latex_audit = f"IEEEtran test return code: {result.returncode}. Overfull diagnostics: {overfull or 'none'}. Font sizes in the source are footnotesize for the resource and residual tables and scriptsize for the complete table."

original_residual = source_tables_df.loc[source_tables_df["input"].eq("Original")].iloc[0]
dimension_lines = "\n".join(
    f"- `{stem}.pdf`: {width:.2f} × {height:.2f} in; matching PNG at 300 dpi."
    for stem, (width, height) in figure_dimensions.items()
)

audit = f'''# Revised article artifacts audit

## Scope and provenance

- Notebook: `{notebook_path.relative_to(REPO_ROOT) if 'notebook_path' in globals() else 'article_artifacts/notebook/article_operational_tables_and_figures.ipynb'}`
- Fixed-FPS raw campaign: `{FIXED_ROOT}`
- Original-trace raw campaign: `{ORIGINAL_ROOT}`
- The revised cells reuse the audited raw-data calculations in this notebook. No raw file, metric definition, run selection, or aggregation rule was changed.
- The experimental replicate is the complete run: five runs per configuration. The fixed campaign contains 8,280 passage-run observations; the full audit contains 9,200 passage-run observations including Original.

## Statistical definitions

- **Mean (temporal resource metrics):** mean of the five within-run means.
- **P95 (temporal resource metrics):** mean of the five within-run P95 values.
- **Mean run maximum:** mean of the five within-run maxima; its 95% Student-t confidence interval uses those five maxima. This term is used only in figures.
- **Abs. peak:** largest observation across all five runs; this is distinct from mean run maximum and is used in tables.
- **Energy/run:** trapezoidal integration of power using real timestamps within each run, reported in kJ; displayed means and 95% Student-t confidence intervals use the five run totals.
- **Post-capture latency:** `weight_prediction_final - last_image_capture_time`. Each latency box contains 184 animal/passage means, each first averaged across five runs.
- **Residual workload:** unchanged audited definitions. For Original, residual passages = {100 * original_residual['residual_passages_mean_fraction']:.1f}%, residual mean = {original_residual['residual_inferences_mean_of_run_means']:.2f}, residual P95 = {original_residual['residual_inferences_mean_of_run_p95']:.2f}, not-started share = {100 * original_residual['not_started_share_mean_fraction']:.1f}%, and drain mean = {fmt_latency(original_residual['drain_mean_of_run_means_s'])} s.

All plotted confidence intervals are 95% Student-t intervals calculated from five complete-run values. Passage observations are not treated as independent treatment replicates.

## Figure presentation

{dimension_lines}

- Font family selected by the notebook: `{SELECTED_FONT}`; fallbacks are STIXGeneral and DejaVu Serif. Tick labels are 8 pt, axis labels 8.5 pt, legends 8 pt, and panel labels 9 pt bold. Lines are 1.4 pt and markers are 4.5 pt.
- PDF font type is 42; PDFs are vector outputs and PNGs are saved at 300 dpi.
- Palette: Mean `#0072B2`; P95 `#E69F00`; Mean run maximum `#D55E00`; grid `#D9D9D9`; box fill `#DCEAF4`; box edge `#0072B2`; median `#222222`; outliers `#777777`.
- The original figures use equally spaced categorical FPS positions; the added `_real_x` figures use numeric FPS coordinates, preserving the actual spacing between configurations.
- Original is excluded from every revised fixed-FPS figure.
- No logarithmic or symlog axis is used. No dual, broken, or hidden auxiliary axis is used.
- CPU uses a linear y axis from 0 to {cpu_ymax:g}% and contains only Mean and P95; no maximum curve is present.
- Temperature uses a linear y axis from {temp_lower:g} to {temp_upper:g} °C; the light gray thermal region begins at 80 °C and extends to the panel top, with a darker gray overlay above 85 °C.
- The real-x CPU/temperature figure uses a temperature y-axis ending at 90 °C so the 85 °C reference and legend remain visually separated.
- Both energy alternatives use the identical linear y axis from 0 to {energy_ymax:g} kJ.
- The additional filled real-x energy point plot uses the same y limits, data, and IC95% bars as the unfilled point plot; only the area from zero to the mean curve is added.
- Energy IC95% half-widths are present for every fixed-FPS point (range {energy_ci_half.min():.4f}--{energy_ci_half.max():.4f} kJ); the revised plots use dark, slightly thicker error bars and caps to make these small run-to-run intervals visible.
- Both latency main panels use a linear absolute y axis from 0 to 110 s with ticks at 0, 20, 40, 60, 80, and 100 s. Outliers are shown. The inset uses 0 to {inset_ymax:g} s, derived from the observed maximum of {inset_ymax_raw:.3f} s for 1–15 FPS.

## Visual comparison and technical recommendation

- **Energy point versus bar:** the point plot is more compact in visual ink and makes the cross-configuration trend and confidence intervals easier to compare in a single IEEE column. The bar plot emphasizes absolute magnitude from zero but is visually denser. The point plot is the stronger compact candidate; the bar plot remains a valid alternative, and the final article choice is intentionally left open.
- **Absolute latency versus inset:** the plain absolute boxplot is simpler and preserves direct comparison on one 0–110 s scale, but the low-FPS distributions are compressed. The inset version retains that full main scale and all outliers while making 1–15 FPS readable. Use the inset candidate when early-regime discrimination is important; use the plain version when minimal visual complexity is preferred.

## Tables and compilation

- `tab_workload_resources.tex` uses `table*`, `tabular*{{\\textwidth}}`, `\\extracolsep{{\\fill}}`, and `\\footnotesize`; it includes temperature P95 and labels campaign-wide extrema as Abs. peak.
- `tab_residual_workload_compact.tex` uses a one-column `tabular*{{\columnwidth}}`. Its note explains fractional P95 values and the interpretation of inferences not yet started at the prediction stage.
- `tab_operational_results_full_revised.tex` is retained as an unscaled alternative and uses Abs. peak terminology. The prior full-table artifact was not deleted.
- {latex_audit}

Because `pdflatex` is unavailable, source font commands and native figure dimensions are reported, but final IEEEtran table fit, effective rendered table font size, overfull boxes, and page-level legibility have not been verified.
'''
(REVISED_ROOT / "revised_artifacts_audit.md").write_text(audit, encoding="utf-8")

display(Markdown(audit))


## Revised artifact inventory

The revised directory contains nine PDF/PNG figure pairs, three LaTeX tables, candidate captions, and a presentation audit. The original artifacts remain available unchanged.

In [ ]:

required_outputs = [
    *(REVISED_FIGURE_DIR / f"{stem}.{suffix}" for stem in FIGURE_SIZES_IN for suffix in ["pdf", "png"]),
    REVISED_TABLE_DIR / "tab_workload_resources.tex",
    REVISED_TABLE_DIR / "tab_residual_workload_compact.tex",
    REVISED_TABLE_DIR / "tab_operational_results_full_revised.tex",
    REVISED_ROOT / "article_figure_captions_revised.tex",
    REVISED_ROOT / "revised_artifacts_audit.md",
]
missing_outputs = [path for path in required_outputs if not path.exists() or path.stat().st_size == 0]
if missing_outputs:
    raise AssertionError(f"Missing revised artifacts: {missing_outputs}")

assert set(FPS_LABELS) == set(run_df.loc[run_df["configuration"].ne("Original"), "configuration"].unique())
assert "Original" not in FPS_LABELS
assert all(len(values) == 184 for values in fixed_latency_data)
assert all(source_tables_df["input"].tolist()[i] == CONFIG_ORDER[i] for i in range(len(CONFIG_ORDER)))

print(f"Validated {len(required_outputs)} revised artifacts.")
for path in required_outputs:
    print(path.relative_to(REPO_ROOT), path.stat().st_size)
